# 01 — Preparação de Dados

Funções: `trata_valores_ausentes`, `verifica_outliers`, `split_dataset`, `codifica_categoricas`, `aplica_encoders`, `compara_modelos_cv`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


## 1. `trata_valores_ausentes`

Preenche NaN de colunas numéricas: **mediana** quando |skew| > 1 (distribuição muito assimétrica) ou **média** quando skew moderado.

In [ ]:
from ML.preparacao_dados import trata_valores_ausentes

np.random.seed(0)
df_nan = pd.DataFrame({
    'preco':    [10, 12, 11, np.nan, 13, 250],        # skew alto (outlier 250) -> mediana
    'desconto': [0.1, 0.2, np.nan, 0.15, 0.18, 0.12], # skew moderado -> media
})

print("Antes:")
print(df_nan)
print(f"\nSkew: {df_nan.skew().to_dict()}")

trata_valores_ausentes(df_nan)

print("\nDepois:")
print(df_nan)


## 2. `verifica_outliers`

Exibe boxplot de cada coluna numérica e retorna as linhas do DataFrame que contêm outliers pela regra IQR.

In [ ]:
from ML.preparacao_dados import verifica_outliers

df_out = pd.DataFrame({
    'vendas':  [100, 110, 105, 98, 112, 108, 850],  # 850 e outlier
    'visitas': [200, 210, 205, 195, 215, 208, 220],
})

df_outliers = verifica_outliers(df_out, distancia_media=1.5)
print(f"\nLinhas com outliers encontradas:\n{df_outliers}")


## 3. `split_dataset`

Divide o dataset em X_train, X_test, y_train, y_test separando a coluna alvo.

In [ ]:
from ML.preparacao_dados import split_dataset

np.random.seed(1)
df_modelo = pd.DataFrame({
    'feature_1': np.random.randn(100),
    'feature_2': np.random.randint(0, 10, 100),
    'target':    np.random.randint(0, 2, 100),
})

X_train, X_test, y_train, y_test = split_dataset(
    data=df_modelo,
    target_column='target',
    test_size=0.2,
    random_state=42,
)

print(f"Treino: X={X_train.shape}, y={y_train.shape}")
print(f"Teste:  X={X_test.shape},  y={y_test.shape}")


## 4. `codifica_categoricas` + `aplica_encoders`

Encoding automático sem *data leakage*: o fit ocorre **apenas no treino**; o teste é transformado com os encoders já ajustados.

| Tipo de coluna | Estratégia |
|---|---|
| Hierarquia definida | `OrdinalEncoder` — ordem que você controla |
| 2 valores únicos | `OneHotEncoder` → 1 coluna binária (drop_first) |
| N valores sem hierarquia | `OneHotEncoder` → N-1 colunas |
| Cardinalidade > max_cats_ohe | Pulada com aviso |

In [ ]:
from ML.preparacao_dados import split_dataset, codifica_categoricas, aplica_encoders

np.random.seed(42)
n = 200
df_enc = pd.DataFrame({
    'genero':        np.random.choice(['M', 'F'], n),
    'canal_venda':   np.random.choice(['Loja', 'Online', 'Televendas', 'Parceiro'], n),
    'nivel_plano':   np.random.choice(['Basico', 'Intermediario', 'Premium'], n),
    'risco_credito': np.random.choice(['Alto', 'Baixo', 'Medio'], n),
    'cidade':        [f'Cidade_{i}' for i in np.random.randint(1, 60, n)],  # alta cardinalidade
    'idade':         np.random.randint(18, 65, n),
    'churn':         np.random.randint(0, 2, n),
})

X_train, X_test, y_train, y_test = split_dataset(df_enc, 'churn', test_size=0.2)

# Fit somente no treino
X_train_enc, encoders = codifica_categoricas(
    df=X_train,
    colunas_ordinais={
        'nivel_plano':   ['Basico', 'Intermediario', 'Premium'],   # 0, 1, 2
        'risco_credito': ['Baixo', 'Medio', 'Alto'],               # 0, 1, 2
    },
    drop_first=True,
    max_cats_ohe=20,
)

# Transform no teste com os mesmos encoders — sem vazar padrao dos dados
X_test_enc = aplica_encoders(X_test, encoders)

print(f"Colunas: {X_train_enc.columns.tolist()}")
print(f"\nColunas treino == Colunas teste: {X_train_enc.columns.tolist() == X_test_enc.columns.tolist()}")
X_train_enc.head()


## 5. `compara_modelos_cv`

Avalia N modelos com validação cruzada e retorna um DataFrame ranqueado por score médio.

- Detecta automaticamente **KFold** (regressão) ou **StratifiedKFold** (classificação) pela cardinalidade de `y`  
- Aceita qualquer métrica sklearn via `scoring`  
- Exibe `média ± desvio` e intervalo `[min, max]` por fold

In [ ]:
from ML.preparacao_dados import compara_modelos_cv
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier

np.random.seed(42)
n = 500
X_cmp = np.random.randn(n, 5)
y_cmp = ((X_cmp[:, 0] + X_cmp[:, 1] * 0.8 - X_cmp[:, 2] * 0.5) > 0).astype(int)

# y com 2 valores unicos -> StratifiedKFold detectado automaticamente
modelos_clf = {
    'Regressao Logistica': LogisticRegression(max_iter=500),
    'Arvore de Decisao':   DecisionTreeClassifier(max_depth=4),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
}

resultado = compara_modelos_cv(modelos_clf, X_cmp, y_cmp, scoring='roc_auc', cv=5)
resultado


In [ ]:
# Exemplo com regressao: KFold detectado automaticamente
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.ensemble import RandomForestRegressor

np.random.seed(0)
X_reg = np.random.randn(300, 4)
y_reg = 3 * X_reg[:, 0] - 1.5 * X_reg[:, 1] + np.random.randn(300) * 0.5

modelos_reg = {
    'Regressao Linear': LinearRegression(),
    'Lasso':            Lasso(alpha=0.1),
    'Random Forest':    RandomForestRegressor(n_estimators=100, random_state=42),
}

resultado_reg = compara_modelos_cv(modelos_reg, X_reg, y_reg, scoring='r2', cv=5)
resultado_reg
